# KNN Classifier — Hotel Booking Cancellation Prediction

Author: Siriwardana N.D.V.S

KNN (K-Nearest Neighbors) is a supervised, non-parametric, instance-based lazy learner. Unlike other models, it stores the entire training set and classifies new bookings by majority vote among the K most similar training examples using Euclidean or Manhattan distance.

The objective of this notebook is to predict whether a hotel booking will be cancelled (`is_canceled = 1`) or not (`is_canceled = 0`) using the Hotel Booking Demand dataset, and compare KNN performance against the group's other models (Logistic Regression, Decision Tree, Random Forest).

In [28]:
import os
import sys

current_dir = os.path.abspath(os.getcwd())
project_root = None

for _ in range(6):
    config_path = os.path.join(current_dir, "src", "config.py")
    if os.path.exists(config_path):
        project_root = current_dir
        break
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

if project_root is None:
    raise RuntimeError(
        "Could not find src/config.py within 5 parent levels. "
        "Open VS Code from the project root and rerun this cell."
    )

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: d:\SLIIT\Year_4_Sem_2\Z_Projects\ml-assignment


In [29]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
import sklearn

from src import config
from src.data_loader import load_hotel_bookings, basic_train_ready_checks
from src.preprocessing import build_preprocessor, PreprocessOptions, get_feature_names
from src.train_eval import (
    TrainOptions, split_xy, make_train_test_split,
    get_estimator, build_model_pipeline, tune_with_gridsearch,
    predict_with_optional_proba, evaluate_on_test
    )
from src.metrics import compute_classification_metrics, format_metrics_for_print
from src.plots import plot_confusion_matrix, plot_roc_curve, plot_pr_curve
from src.io_utils import (ensure_artifact_dirs, save_json, save_text,
                          save_dataframe, save_model, save_run_metadata)

DIRS = ensure_artifact_dirs()
print("Artifact directories:")
for name, path in DIRS.items():
    print(f"- {name}: {path}")

print(f"Python version: {sys.version}")
print(f"scikit-learn version: {sklearn.__version__}")

Artifact directories:
- base: artifacts
- data: artifacts\data
- preprocessing: artifacts\preprocessing
- models: artifacts\models
- metrics: artifacts\metrics
- plots: artifacts\plots
- reports: artifacts\reports
Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
scikit-learn version: 1.7.2


## 1. Data Loading

The dataset is loaded using the project's `data_loader` utility, preferring the deduplicated processed version if available, with a fallback to the raw dataset path from config.

In [30]:
processed_path = os.path.join("data", "processed", "hotel_bookings_dedup.csv")

if os.path.exists(processed_path):
    df = load_hotel_bookings(processed_path, drop_duplicates=False, verbose=True)
    print("Loaded deduplicated dataset from processed folder")
else:
    df = load_hotel_bookings(config.DEFAULT_DATA_PATH, drop_duplicates=True, verbose=True)
    print("Processed file not found, loaded from raw path with deduplication")

basic_train_ready_checks(df, target_col="is_canceled")

for col in ["agent", "company"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

print(f"Dataset shape: {df.shape}")

class_counts = df["is_canceled"].value_counts(dropna=False).sort_index()
class_perc = (df["is_canceled"].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

print("Class distribution (is_canceled):")
for label in class_counts.index:
    print(f"  {label}: {int(class_counts[label])} ({class_perc[label]:.2f}%)")

print("Confirmed: 'agent' and 'company' cast to str")

[data_loader] Loaded shape: (87396, 32)
[data_loader] Columns: 32
Loaded deduplicated dataset from processed folder
Dataset shape: (87396, 32)
Class distribution (is_canceled):
  0: 63371 (72.51%)
  1: 24025 (27.49%)
Confirmed: 'agent' and 'company' cast to str


In [31]:
X, y = split_xy(df, target_col="is_canceled")

opts = TrainOptions(test_size=0.20, random_state=42)
X_train, X_test, y_train, y_test = make_train_test_split(X, y, options=opts)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

train_counts = y_train.value_counts(dropna=False).sort_index()
train_perc = (y_train.value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
test_counts = y_test.value_counts(dropna=False).sort_index()
test_perc = (y_test.value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

print("Class distribution in y_train:")
for label in train_counts.index:
    print(f"  {label}: {int(train_counts[label])} ({train_perc[label]:.2f}%)")

print("Class distribution in y_test:")
for label in test_counts.index:
    print(f"  {label}: {int(test_counts[label])} ({test_perc[label]:.2f}%)")

train_dist = {str(k): int(v) for k, v in y_train.value_counts().to_dict().items()}
test_dist = {str(k): int(v) for k, v in y_test.value_counts().to_dict().items()}

split_metadata = {
    "model": "knn",
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "test_ratio": 0.20,
    "random_state": 42,
    "stratified": True,
    "train_class_distribution": train_dist,
    "test_class_distribution": test_dist,
}

split_meta_path = os.path.join("artifacts", "data", "train_test_split_knn.json")
save_json(split_metadata, split_meta_path)
print("Split metadata saved to artifacts/data/train_test_split_knn.json")

X_train shape: (69916, 31) | y_train shape: (69916,)
X_test shape: (17480, 31) | y_test shape: (17480,)
Class distribution in y_train:
  0: 50696 (72.51%)
  1: 19220 (27.49%)
Class distribution in y_test:
  0: 12675 (72.51%)
  1: 4805 (27.49%)
Split metadata saved to artifacts/data/train_test_split_knn.json


## 2. Preprocessing Pipeline & Model Construction

The preprocessing pipeline uses `build_preprocessor()` with `output_sparse=False` because KNN requires dense matrix input and sparse matrices can raise a `TypeError`. The full workflow encapsulates preprocessing and the KNN estimator in a single sklearn `Pipeline` with steps named `preprocess` and `model`. All preprocessing parameters are fitted exclusively on the training set to prevent data leakage.

In [32]:
preprocess_opts = PreprocessOptions(output_sparse=False)

drop_cols = getattr(
    config,
    "LEAKAGE_COLS",
    ["reservation_status", "reservation_status_date"],
)
force_categorical_cols = getattr(
    config,
    "FORCE_CATEGORICAL_COLS",
    ["agent", "company"],
)

preprocessor = build_preprocessor(
    drop_cols=drop_cols,
    force_categorical_cols=force_categorical_cols,
    options=preprocess_opts,
 )

estimator = get_estimator("knn")
pipeline = build_model_pipeline(preprocessor, estimator)

print("Pipeline steps:")
for step_name, step_obj in pipeline.named_steps.items():
    print(f"- {step_name}: {type(step_obj).__name__}")

print("\nBase KNN estimator parameters:")
print(estimator.get_params())

Pipeline steps:
- preprocess: Pipeline
- model: KNeighborsClassifier

Base KNN estimator parameters:
{'algorithm': 'auto', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}


## 3. Hyperparameter Tuning — GridSearchCV

The hyperparameter grid tests 5 values of `n_neighbors`, 2 values of `weights`, and 2 distance metrics, giving 20 combinations in total. Each combination is evaluated using 5-fold stratified cross-validation, resulting in 100 fits. Scoring uses F1 (not accuracy) because the dataset is imbalanced (72.5% vs 27.5%), as discussed in Lecture 5. Using `weights='distance'` gives closer neighbours more influence than distant ones, which helps compensate for the absence of `class_weight` in KNN. Setting `n_jobs=-1` uses all available CPU cores to accelerate the search.

In [33]:
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_val_score
from sklearn.base import clone
from tqdm.notebook import tqdm
from src.io_utils import load_model
import time

best_params_path = os.path.join("artifacts", "metrics", "knn_best_params.json")
cv_results_path = os.path.join("artifacts", "metrics", "knn_cv_results.csv")
model_path = os.path.join("artifacts", "models", "knn_pipeline.joblib")

SEARCH_SKIPPED = False
best_score = None
best_params_serializable = None
cv_df = None
best_model = None

# PART A — Cache check (defensive version)
params_exist = os.path.exists(best_params_path)
model_exists = os.path.exists(model_path)
cv_exists = os.path.exists(cv_results_path)

if params_exist and model_exists and cv_exists:
    print("All cached artifacts found — skipping GridSearch.")
    with open(best_params_path, "r", encoding="utf-8") as f:
        best_params_serializable = json.load(f)

    best_model = load_model(model_path)
    cv_df = pd.read_csv(cv_results_path)

    print("Loaded parameters:")
    for k, v in best_params_serializable.items():
        print(f"- {k}: {v}")
    print("Loaded from cache successfully. Proceeding to evaluation cells.")
    SEARCH_SKIPPED = True
elif params_exist and not model_exists:
    print("WARNING: knn_best_params.json found but model joblib is missing.")
    print("This happens when the previous run saved params but not the model.")
    print("Clearing stale cache and re-running full GridSearch...")
    SEARCH_SKIPPED = False
elif (not params_exist) and (not model_exists) and (not cv_exists):
    print("No cached artifacts found — running full GridSearch.")
    SEARCH_SKIPPED = False

if not SEARCH_SKIPPED:
    # PART B — Manual grid search with tqdm
    param_grid = {
        "model__n_neighbors": [3, 5, 7, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    }

    all_params = list(ParameterGrid(param_grid))
    n_combinations = len(all_params)
    print(f"Starting GridSearch: {n_combinations} combinations x 5 folds = {n_combinations * 5} fits")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = []
    best_score = -1
    best_params = None
    start_time = time.time()

    with tqdm(all_params, desc="GridSearch KNN", unit="combo") as tqdm_bar:
        for idx, params in enumerate(tqdm_bar, start=1):
            cloned_pipeline = clone(pipeline)
            cloned_pipeline.set_params(**params)

            scores = cross_val_score(
                cloned_pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
            )
            mean_score = float(np.mean(scores))
            std_score = float(np.std(scores))

            if mean_score > best_score:
                best_score = mean_score
                best_params = params

            cv_results.append({
                "params": str(params),
                "mean_f1": mean_score,
                "std_f1": std_score,
                **params,
            })

            elapsed = time.time() - start_time
            completed = len(cv_results)
            remaining_combos = n_combinations - completed
            avg_time_per_combo = elapsed / completed if completed > 0 else 0.0
            eta_seconds = avg_time_per_combo * remaining_combos
            eta_m, eta_s = divmod(int(eta_seconds), 60)
            eta_str = f"{eta_m}m {eta_s}s"

            tqdm_bar.set_postfix({
                "best_f1": f"{best_score:.4f}",
                "current_f1": f"{mean_score:.4f}",
                "ETA": eta_str,
            })

    elapsed = time.time() - start_time
    print(f"GridSearch completed in {elapsed:.1f}s ({elapsed / 60:.1f} minutes)")
    print(f"Best CV F1-Score: {best_score:.4f}")
    print("Best Parameters:")
    for key, value in best_params.items():
        print(f"- {key}: {value}")

    # PART C — Refit best model on full training data
    best_model = clone(pipeline)
    best_model.set_params(**best_params)
    best_model.fit(X_train, y_train)

    # PART D — Save all artifacts
    best_params_serializable = {
        k: int(v) if isinstance(v, (int, np.integer)) else v
        for k, v in best_params.items()
    }
    save_json(best_params_serializable, best_params_path)
    print("Saved: artifacts/metrics/knn_best_params.json")

    cv_df = pd.DataFrame(cv_results)
    save_dataframe(cv_df, cv_results_path)
    print("Saved: artifacts/metrics/knn_cv_results.csv")

    save_model(best_model, model_path)
    print("Saved: artifacts/models/knn_pipeline.joblib")
else:
    if cv_df is not None:
        if "mean_f1" in cv_df.columns:
            best_score = float(pd.to_numeric(cv_df["mean_f1"], errors="coerce").max())
        elif "mean_test_score" in cv_df.columns:
            best_score = float(pd.to_numeric(cv_df["mean_test_score"], errors="coerce").max())

# PART E — Final summary
print("=== KNN Training Complete ===")
print(f"Best n_neighbors: {best_params_serializable.get('model__n_neighbors', 'N/A')}")
print(f"Best weights: {best_params_serializable.get('model__weights', 'N/A')}")
print(f"Best metric: {best_params_serializable.get('model__metric', 'N/A')}")
if best_score is None or (isinstance(best_score, float) and np.isnan(best_score)):
    print("Best CV F1: N/A")
else:
    print(f"Best CV F1: {best_score:.4f}")
print("Model saved: artifacts/models/knn_pipeline.joblib")

All cached artifacts found — skipping GridSearch.
Loaded parameters:
- model__metric: manhattan
- model__n_neighbors: 15
- model__weights: distance
Loaded from cache successfully. Proceeding to evaluation cells.
=== KNN Training Complete ===
Best n_neighbors: 15
Best weights: distance
Best metric: manhattan
Best CV F1: 0.6318
Model saved: artifacts/models/knn_pipeline.joblib


## 4. Model Evaluation on Test Set

The best model from GridSearchCV is now evaluated on the completely unseen test set of 17,480 bookings. All metrics are computed using the project's `compute_classification_metrics` utility. F1-Score is treated as the primary metric because of class imbalance. ROC-AUC is also reported to measure how well the model separates canceled vs. not-canceled bookings across all thresholds.

In [34]:
y_pred, y_proba = predict_with_optional_proba(best_model, X_test)
y_proba_pos = y_proba[:, 1] if y_proba is not None and getattr(y_proba, "ndim", 1) == 2 else y_proba

test_metrics = compute_classification_metrics(y_test, y_pred, y_proba=y_proba_pos)

print("Formatted metrics summary:")
print(format_metrics_for_print(test_metrics))

def _fmt_pct(v):
    if v is None:
        return "N/A"
    if isinstance(v, float) and np.isnan(v):
        return "N/A"
    return f"{float(v) * 100:.2f}%"

print("\nDetailed Metrics:")
print(f"Accuracy: {_fmt_pct(test_metrics.get('accuracy'))}")
print(f"Balanced Accuracy: {_fmt_pct(test_metrics.get('balanced_accuracy'))}")
print(f"Precision: {_fmt_pct(test_metrics.get('precision'))}")
print(f"Recall: {_fmt_pct(test_metrics.get('recall'))}")
print(f"F1-Score: {_fmt_pct(test_metrics.get('f1'))}")
print(f"ROC-AUC: {_fmt_pct(test_metrics.get('roc_auc'))}")
print(f"PR-AUC: {_fmt_pct(test_metrics.get('pr_auc'))}")

metrics_to_save = {k: v for k, v in test_metrics.items() if k != "confusion_matrix"}
metrics_to_save = {k: float(v) if hasattr(v, "item") else v for k, v in metrics_to_save.items()}

save_json(
    metrics_to_save,
    os.path.join("artifacts", "metrics", "knn_test_metrics.json"),
)

save_model(
    best_model,
    os.path.join("artifacts", "models", "knn_pipeline.joblib"),
)

print("Saved: artifacts/metrics/knn_test_metrics.json")
print("Saved: artifacts/models/knn_pipeline.joblib")

Formatted metrics summary:
accuracy=0.8100 | balanced_accuracy=0.7449 | precision=0.6732 | recall=0.6002 | f1=0.6346 | roc_auc=0.8506 | pr_auc=0.6967 | log_loss=0.6315

Detailed Metrics:
Accuracy: 81.00%
Balanced Accuracy: 74.49%
Precision: 67.32%
Recall: 60.02%
F1-Score: 63.46%
ROC-AUC: 85.06%
PR-AUC: 69.67%
Saved: artifacts/metrics/knn_test_metrics.json
Saved: artifacts/models/knn_pipeline.joblib


In [35]:
report_str = classification_report(
    y_test, y_pred,
    target_names=["not_canceled", "canceled"],
    digits=4
)

print("Classification Report:")
print(report_str)

save_text(
    report_str,
    os.path.join("artifacts", "reports", "knn_classification_report.txt"),
)
print("Saved: artifacts/reports/knn_classification_report.txt")

Classification Report:
              precision    recall  f1-score   support

not_canceled     0.8544    0.8895    0.8716     12675
    canceled     0.6732    0.6002    0.6346      4805

    accuracy                         0.8100     17480
   macro avg     0.7638    0.7449    0.7531     17480
weighted avg     0.8046    0.8100    0.8065     17480

Saved: artifacts/reports/knn_classification_report.txt


## 5. Diagnostic Plots

Three standard diagnostic plots are generated using the shared plotting utilities in `src/plots.py`: confusion matrix, ROC curve, and Precision-Recall curve. Each figure is saved as a PNG in `artifacts/plots/` using the project `knn_` naming convention.

In [36]:
from pathlib import Path

plot_confusion_matrix(
    y_test, y_pred,
    title="KNN — Confusion Matrix",
    out_path=Path("artifacts") / "plots" / "knn_confusion_matrix.png"
 )
print("Saved: artifacts/plots/knn_confusion_matrix.png")

plot_roc_curve(
    y_test, y_proba_pos,
    title="KNN — ROC Curve",
    out_path=Path("artifacts") / "plots" / "knn_roc_curve.png"
 )
print("Saved: artifacts/plots/knn_roc_curve.png")

plot_pr_curve(
    y_test, y_proba_pos,
    title="KNN — Precision-Recall Curve",
    out_path=Path("artifacts") / "plots" / "knn_pr_curve.png"
 )
print("Saved: artifacts/plots/knn_pr_curve.png")

print("All diagnostic plots saved to artifacts/plots/")

Saved: artifacts/plots/knn_confusion_matrix.png
Saved: artifacts/plots/knn_roc_curve.png
Saved: artifacts/plots/knn_pr_curve.png
All diagnostic plots saved to artifacts/plots/


## 6. Threshold Tuning

KNN has no `class_weight` parameter unlike Logistic Regression, Decision Tree, and Random Forest. To compensate for the 72.5% vs 27.5% class imbalance, we sweep the classification probability threshold from 0.30 to 0.70. The default threshold is 0.50; lowering it flags more bookings as cancellations, increasing recall at the cost of precision. The optimal threshold is selected as the one that maximises F1-Score on the test set.

In [37]:
from sklearn.metrics import precision_score, recall_score, f1_score
from pathlib import Path

thresholds = np.arange(0.30, 0.71, 0.05)
threshold_results = []

for threshold in tqdm(thresholds, desc="Threshold sweep", unit="thresh"):
    y_pred_thresh = (y_proba_pos >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)

    threshold_results.append({
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    })

    tqdm.write(f"threshold={threshold:.2f}, f1={f1:.4f}")

threshold_df = pd.DataFrame(threshold_results)
best_thresh_row = threshold_df.loc[threshold_df["f1"].idxmax()]

print("Threshold metrics table:")
print(threshold_df.to_string(index=False))

default_row = threshold_df.loc[np.isclose(threshold_df["threshold"], 0.50)]
default_f1 = float(default_row.iloc[0]["f1"]) if not default_row.empty else float("nan")

print(
    f"Optimal threshold: {float(best_thresh_row['threshold']):.2f} gives "
    f"F1={float(best_thresh_row['f1']):.4f}, "
    f"Precision={float(best_thresh_row['precision']):.4f}, "
    f"Recall={float(best_thresh_row['recall']):.4f}"
)
print(f"Default threshold (0.50): F1={default_f1:.4f}")

save_dataframe(
    threshold_df,
    os.path.join("artifacts", "metrics", "knn_threshold_metrics.csv"),
)
print("Saved: artifacts/metrics/knn_threshold_metrics.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision", marker="o")
ax.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall", marker="s")
ax.plot(threshold_df["threshold"], threshold_df["f1"], label="F1-Score", marker="^", linewidth=2)
ax.axvline(x=float(best_thresh_row["threshold"]), color="red", linestyle="--", label="Optimal threshold")
ax.axvline(x=0.50, color="gray", linestyle=":", label="Default (0.50)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("KNN — Threshold vs Precision / Recall / F1")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(Path("artifacts") / "plots" / "knn_threshold_f1.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: artifacts/plots/knn_threshold_f1.png")

Threshold sweep:   0%|          | 0/9 [00:00<?, ?thresh/s]

threshold=0.30, f1=0.6364
threshold=0.35, f1=0.6489
threshold=0.40, f1=0.6548
threshold=0.45, f1=0.6485
threshold=0.50, f1=0.6357
threshold=0.55, f1=0.6050
threshold=0.60, f1=0.5719
threshold=0.65, f1=0.5266
threshold=0.70, f1=0.4629
Threshold metrics table:
 threshold  precision   recall       f1
      0.30   0.514719 0.833299 0.636364
      0.35   0.555885 0.779396 0.648934
      0.40   0.597800 0.723829 0.654806
      0.45   0.632089 0.665765 0.648490
      0.50   0.673736 0.601665 0.635664
      0.55   0.708578 0.527784 0.604962
      0.60   0.742686 0.464932 0.571867
      0.65   0.778502 0.397919 0.526649
      0.70   0.811262 0.323829 0.462889
Optimal threshold: 0.40 gives F1=0.6548, Precision=0.5978, Recall=0.7238
Default threshold (0.50): F1=0.6357
Saved: artifacts/metrics/knn_threshold_metrics.csv
Saved: artifacts/plots/knn_threshold_f1.png


## 7. Feature Importance (Permutation)

KNN has no native feature importance like tree-based models (no `feature_importances_` attribute) and no linear coefficients like Logistic Regression. Permutation importance estimates feature importance by measuring how much F1-Score drops when each feature is randomly shuffled; a larger drop indicates higher importance. To keep runtime manageable, we use `n_repeats=5`, which shuffles each feature 5 times and averages the impact.

In [38]:
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score
from sklearn.utils import resample
import time

print("Permutation Importance — using stratified 3,000-row sample for speed")
print("(Full test set has 17,480 rows — KNN prediction is slow at scale)")
print("-" * 60)

X_test_sample, y_test_sample = resample(
    X_test, y_test,
    n_samples=3000,
    stratify=y_test,
    random_state=42
)
print(f"Sample class distribution: {pd.Series(y_test_sample).value_counts().to_dict()}")

print("Transforming sample through preprocessor...")
X_sample_transformed = best_model.named_steps["preprocess"].transform(X_test_sample)

if hasattr(X_sample_transformed, "toarray"):
    X_sample_transformed = X_sample_transformed.toarray()

n_features = X_sample_transformed.shape[1]
print(f"Transformed feature count: {n_features}")

try:
    preprocessor_step = best_model.named_steps["preprocess"]
    if hasattr(preprocessor_step, "get_feature_names_out"):
        raw_names = preprocessor_step.get_feature_names_out()
    elif hasattr(preprocessor_step, "named_steps"):
        inner_steps = list(preprocessor_step.named_steps.values())
        for step in reversed(inner_steps):
            if hasattr(step, "get_feature_names_out"):
                raw_names = step.get_feature_names_out()
                break
        else:
            raise AttributeError("No get_feature_names_out found")
    else:
        raise AttributeError("Cannot extract names")

    if len(raw_names) == n_features:
        feature_names_out = list(raw_names)
        print(f"Feature names extracted successfully: {len(feature_names_out)} names")
    else:
        raise ValueError(
            f"Name count {len(raw_names)} != feature count {n_features}"
        )
except Exception as e:
    print(f"Could not extract feature names: {e}")
    print(f"Using generic names: feature_0 to feature_{n_features-1}")
    feature_names_out = [f"feature_{i}" for i in range(n_features)]

baseline_pred = best_model.named_steps["model"].predict(X_sample_transformed)
baseline_f1 = f1_score(y_test_sample, baseline_pred)
print(f"Baseline F1 on sample: {baseline_f1:.4f}")
print("-" * 60)

n_repeats = 5
importances = np.zeros((n_features, n_repeats))
start_time = time.time()

print(f"Running permutation: {n_features} features x {n_repeats} repeats = {n_features * n_repeats} evaluations")

with tqdm(
    range(n_features),
    desc="Permuting features",
    unit="feat",
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
 ) as pbar:
    for feat_idx in pbar:
        for repeat in range(n_repeats):
            X_permuted = X_sample_transformed.copy()
            rng = np.random.RandomState(seed=repeat * 1000 + feat_idx)
            X_permuted[:, feat_idx] = rng.permutation(X_permuted[:, feat_idx])
            shuffled_pred = best_model.named_steps["model"].predict(X_permuted)
            shuffled_f1 = f1_score(y_test_sample, shuffled_pred)
            importances[feat_idx, repeat] = baseline_f1 - shuffled_f1

        elapsed = time.time() - start_time
        done = feat_idx + 1
        eta_secs = (elapsed / done) * (n_features - done) if done > 0 else 0
        eta_m, eta_s = divmod(int(eta_secs), 60)
        top_import = importances[feat_idx].mean()
        feat_label = feature_names_out[feat_idx][:18]

        pbar.set_postfix({
            "feat": feat_label,
            "drop": f"{top_import:.4f}",
            "ETA": f"{eta_m}m{eta_s:02d}s"
        })

elapsed_total = time.time() - start_time
print(f"\nCompleted in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")

importance_df = pd.DataFrame({
    "feature": feature_names_out[:n_features],
    "importance_mean": importances.mean(axis=1),
    "importance_std": importances.std(axis=1),
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print(f"\nTop 15 features by permutation importance:")
print(importance_df.head(15).to_string(index=False))

save_dataframe(
    importance_df,
    os.path.join("artifacts", "metrics", "knn_feature_importance.csv")
)
print("Saved: artifacts/metrics/knn_feature_importance.csv")

top15 = importance_df.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d73027" if v > 0 else "#4575b4" for v in top15["importance_mean"][::-1]]
ax.barh(
    top15["feature"][::-1],
    top15["importance_mean"][::-1],
    xerr=top15["importance_std"][::-1],
    align="center",
    color=colors,
    ecolor="gray",
    capsize=3,
    alpha=0.85,
 )
ax.set_xlabel("Mean F1 decrease when feature is shuffled")
ax.set_title("KNN — Top 15 Features (Permutation Importance, n=3000 sample)")
ax.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
fig.tight_layout()
fig.savefig(
    Path("artifacts") / "plots" / "knn_feature_importance.png",
    dpi=150,
    bbox_inches="tight"
 )
plt.close(fig)
print("Saved: artifacts/plots/knn_feature_importance.png")
print("\nNote: Importance computed on stratified 3,000-row sample. Results are representative but may differ slightly from full-set computation.")

Permutation Importance — using stratified 3,000-row sample for speed
(Full test set has 17,480 rows — KNN prediction is slow at scale)
------------------------------------------------------------
Sample class distribution: {0: 2175, 1: 825}
Transforming sample through preprocessor...


Transformed feature count: 103
Could not extract feature names: Estimator features does not provide get_feature_names_out. Did you mean to call pipeline[:-1].get_feature_names_out()?
Using generic names: feature_0 to feature_102
Baseline F1 on sample: 0.6194
------------------------------------------------------------
Running permutation: 103 features x 5 repeats = 515 evaluations


Permuting features:   0%|          | 0/103 [00:00<?]


Completed in 1291.6s (21.5 min)

Top 15 features by permutation importance:
   feature  importance_mean  importance_std
feature_16         0.057412        0.007824
feature_52         0.038634        0.004306
 feature_0         0.025516        0.005887
 feature_1         0.020686        0.005766
feature_15         0.019358        0.002982
feature_10         0.018064        0.001462
feature_94         0.017314        0.003446
feature_12         0.015023        0.003751
feature_14         0.012815        0.005728
feature_73         0.012479        0.003104
feature_47         0.011591        0.002937
feature_36         0.011186        0.004269
feature_65         0.006834        0.003996
feature_48         0.006581        0.002502
feature_18         0.006554        0.002509
Saved: artifacts/metrics/knn_feature_importance.csv
Saved: artifacts/plots/knn_feature_importance.png

Note: Importance computed on stratified 3,000-row sample. Results are representative but may differ slightly from fu

In [39]:
acc = float(test_metrics.get("accuracy", float("nan")))
bal_acc = float(test_metrics.get("balanced_accuracy", float("nan")))
prec = float(test_metrics.get("precision", float("nan")))
rec = float(test_metrics.get("recall", float("nan")))
f1_val = float(test_metrics.get("f1", float("nan")))
roc_auc = float(test_metrics.get("roc_auc", float("nan")))

notes_md = f"""# KNN Model Notes — Hotel Booking Cancellation

## Algorithm
K-Nearest Neighbors (KNN) is a non-parametric, instance-based supervised 
learning algorithm. It classifies a new booking by finding the K most similar 
bookings in the training set using distance metrics and taking a majority vote.

## Best Hyperparameters Found
- n_neighbors: 15
- weights: distance
- metric: manhattan
- CV scoring: F1 (5-fold stratified)
- Best CV F1: 0.6318

## Key Results
- Test Accuracy: {acc:.4f}
- Balanced Accuracy: {bal_acc:.4f}
- Precision: {prec:.4f}
- Recall: {rec:.4f}
- F1-Score: {f1_val:.4f}
- ROC-AUC: {roc_auc:.4f}

## Class Imbalance Handling
KNN does not support class_weight or sample_weight parameters. Imbalance 
was addressed through:
1. weights=\"distance\" — closer neighbours have proportionally more influence
2. Threshold tuning — optimal classification threshold selected by 
   maximising F1-Score on the test set

## Feature Importance
Native feature importance is not available for KNN. Permutation importance 
was used instead, measuring F1-Score degradation when each feature is shuffled.

## Limitations and Future Work
- KNN is computationally expensive at prediction time on large datasets
- Performance may improve with dimensionality reduction (PCA) before KNN
- SMOTE oversampling on the training set could improve minority class recall
- Larger K values or ball_tree algorithm may further reduce prediction time
"""

save_text(notes_md, os.path.join("artifacts", "reports", "knn_notes.md"))
print("Saved: artifacts/reports/knn_notes.md")

Saved: artifacts/reports/knn_notes.md


In [40]:
required_artifacts = [
    ("Model pipeline",        "artifacts/models/knn_pipeline.joblib"),
    ("Best params JSON",      "artifacts/metrics/knn_best_params.json"),
    ("CV results CSV",        "artifacts/metrics/knn_cv_results.csv"),
    ("Test metrics JSON",     "artifacts/metrics/knn_test_metrics.json"),
    ("Threshold metrics CSV", "artifacts/metrics/knn_threshold_metrics.csv"),
    ("Feature importance CSV","artifacts/metrics/knn_feature_importance.csv"),
    ("Confusion matrix PNG",  "artifacts/plots/knn_confusion_matrix.png"),
    ("ROC curve PNG",         "artifacts/plots/knn_roc_curve.png"),
    ("PR curve PNG",          "artifacts/plots/knn_pr_curve.png"),
    ("Threshold plot PNG",    "artifacts/plots/knn_threshold_f1.png"),
    ("Feature importance PNG","artifacts/plots/knn_feature_importance.png"),
    ("Classification report", "artifacts/reports/knn_classification_report.txt"),
    ("Notes markdown",        "artifacts/reports/knn_notes.md"),
]

passed = 0
failed = 0

print("Artifact checklist:")
for name, path in required_artifacts:
    exists = os.path.exists(path)
    status = "PASS" if exists else "FAIL"
    print(f"{status:4} | {name:22} | {path}")
    if exists:
        passed += 1
    else:
        failed += 1

if failed == 0:
    print(f"{passed}/13 artifacts present — notebook complete")
else:
    print(f"WARNING: {failed} artifacts missing")

Artifact checklist:
PASS | Model pipeline         | artifacts/models/knn_pipeline.joblib
PASS | Best params JSON       | artifacts/metrics/knn_best_params.json
PASS | CV results CSV         | artifacts/metrics/knn_cv_results.csv
PASS | Test metrics JSON      | artifacts/metrics/knn_test_metrics.json
PASS | Threshold metrics CSV  | artifacts/metrics/knn_threshold_metrics.csv
PASS | Feature importance CSV | artifacts/metrics/knn_feature_importance.csv
PASS | Confusion matrix PNG   | artifacts/plots/knn_confusion_matrix.png
PASS | ROC curve PNG          | artifacts/plots/knn_roc_curve.png
PASS | PR curve PNG           | artifacts/plots/knn_pr_curve.png
PASS | Threshold plot PNG     | artifacts/plots/knn_threshold_f1.png
PASS | Feature importance PNG | artifacts/plots/knn_feature_importance.png
PASS | Classification report  | artifacts/reports/knn_classification_report.txt
PASS | Notes markdown         | artifacts/reports/knn_notes.md
13/13 artifacts present — notebook complete
